In [ ]:
import json, random, time
from IPython.display import clear_output # le dice directamente a jupyter que limpie la celda, sin necesidad de depender del sistema operativo

#Constantes--------------------------------------------------
LETRAS_VALIDAS = "qwertyuiopasdfghjklñzxcvbnmáéíóú"
MAX_ERRORES = 10
TITULO = r"""
 ██╗  ██╗ █████╗ ███╗   ██╗ ██████╗  █████╗ ███╗   ██╗
 ██║  ██║██╔══██╗████╗  ██║██╔════╝ ██╔══██╗████╗  ██║
 ███████║███████║██╔██╗ ██║██║  ███╗███████║██╔██╗ ██║
 ██╔══██║██╔══██║██║╚██╗██║██║   ██║██╔══██║██║╚██╗██║
 ██║  ██║██║  ██║██║ ╚████║╚██████╔╝██║  ██║██║ ╚████║
 ╚═╝  ╚═╝╚═╝  ╚═╝╚═╝  ╚═══╝ ╚═════╝ ╚═╝  ╚═╝╚═╝  ╚═══╝
"""
#Constantes--------------------------------------------------


def archivos():
    with open("spanish.json", "r", encoding="utf-8") as f:  
        data = json.load(f) # Abre y carga los datos json (cierra el archivo automaticamente) 
    with open("hangman.json", "r", encoding="utf-8") as f:
        ascii_art = json.load(f) # Abre y carga los datos, pero el resultado queda en el diccionario. 


    # Verificación para que hangman.json tenga las suficientes etapas antes de empezar.
    etapas = ascii_art["Hangman"] # Atajo para no escribir ascii_art["Hangman"] cada vez.
    if len(etapas) < MAX_ERRORES + 1: # Verifica que la lista tenga 11 elementos. 
        raise ValueError( # Si esto se cumple mostrar error. 
            f"hangman.json tiene solo {len(etapas)} etapas en 'Hangman', "
            f"pero se necesitan {MAX_ERRORES + 1} (índices 0 a {MAX_ERRORES})."
        )
    
    return data, etapas

def imprimir(etapas, errores, tabla):
    clear_output(wait=True) # Imprime el nuevo estado del juego y elimina el anterior, funciona como loop.
    time.sleep(0.05) #Descubrimos que a veces falla el clear_output si va seguido de un input, por tanto, le daremos un tiempo para evitar el problema
    print(etapas[errores])
    linea = ' '.join(tabla) # Une los elementos de la tabla con separación de un espacio 
    print(f"\n  {linea}\n") 
    print(f"  Intentos fallidos  : {errores}/{MAX_ERRORES}")
    print(f"  Intentos restantes : {MAX_ERRORES - errores}")

    return linea


def normalizar(c): 
    return 'a' if c=='á' else 'e' if c=='é' else 'i' if c=='í' else 'o' if c=='ó' else 'u' if c=='ú' else c # Valida letras normales y sus tildes.

def valida(usadas,etapas,errores,tabla): # Función que valida que la letra del jugador sea correcta antes de ser utilizada en el juego. 
    while True:
        c = input("Adivinanza: ").lower().strip() 

        if len(c) != 1: # ¿Qué ocurre si el jugador ingresó mas de un carácter? 
            imprimir(etapas,errores,tabla)
            print("Error: ingresa solo UNA letra a la vez.") # Mostrar error
            continue
        if c not in LETRAS_VALIDAS: # ¿Qué ocurre si la letra no está en el abecedario?
            imprimir(etapas,errores,tabla)
            print(f"Error: '{c}' no es una letra válida del abecedario.") # Imprimir error
            continue
        cn = normalizar(c) # Primero normaliza la letra (saca la tilde). 
        if cn in usadas: 
            imprimir(etapas,errores,tabla)
            print(f"Error: la letra '{c}' ya fue ingresada antes.") # luego revisa si fue usada antes, en caso de ser así muestra error.
            continue

        return cn

def terminal(etapas,real):
    palabra_norm = ''.join(normalizar(x) for x in real) # Lee las palabras sin tilde, las normaliza y las une en un string. 
    tabla = ['_'] * len(real) # Se reemplaza el guión por la letra una vez que la letra esté correcta.
    usadas      = set() # En un conjunto vacío se guardan todas las letras ya ingresadas.
    incorrectas = [] # Guarda todos los errores.  
    errores     = 0 
    while True:
        linea = imprimir(etapas,errores,tabla)

        if '_' not in linea: # Si todas las lineas estan utilizadas por letras reales...
            while True:
                clear_output(wait=True)
                print("\n  ¡Ganaste! ¡Adivinaste la palabra!\n")
                res = input(f"\n\n ¿Continuar? (S/N)")
                if res.lower() == 's': break
                elif res.lower() == 'n': exit()
            break
        if errores >= MAX_ERRORES: # LLegas a los 10 intentos fallidos.
            while True:
                clear_output(wait=True)
                print(f"\n  ¡Perdiste! La palabra correcta era: {real.capitalize()}\n")
                res = input(f"\n\n ¿Continuar? (S/N)")
                if res.lower() == 's': break
                elif res.lower() == 'n': exit()
            break
        if incorrectas:
            print(f"  Letras incorrectas : {', '.join(incorrectas)}")

        print()
        caracter = valida(usadas,etapas,errores,tabla)
        usadas.add(caracter)

        if caracter in palabra_norm: # Revela la posicion y la letra a la vez. 
            for i, x in enumerate(real): 
                if caracter == normalizar(x):
                    tabla[i] = x
        else:
            errores += 1
            incorrectas.append(caracter)


def consola():    
    data,etapas = archivos()
    palabras_agregadas = set()
    pantalla = 0
    while True:
        if pantalla == 0:
            clear_output(wait=True)
            time.sleep(0.05)
            print(f"{TITULO}\n\n")
            print("1) Jugar")
            print("2) Agregar palabras")
            print("3) Intrucciones")
            print("4) Salir")
            while not (1<=pantalla<=4):
                pantalla = int(input("Opción: "))


        elif pantalla == 1:
            clear_output(wait=True)
            time.sleep(0.05)
            print("Selección Set de Palabras")
            print("1) Integrado")
            print("2) Agregado") if palabras_agregadas else print("2) NO HAY PALABRAS AGREGADAS")
            print("3) Ambos")
            print("4) Atras")
            while not (10<=pantalla<=40):
                pantalla = (int(input("Opción: ")))*10

            if pantalla == 10: real = random.choice(data).lower()
            elif pantalla == 20 and palabras_agregadas: real = random.choice(list(palabras_agregadas)).lower()
            elif pantalla == 20 and not palabras_agregadas: pantalla = 0 
            elif pantalla == 30: real = random.choice(list(data) + list(palabras_agregadas)).lower()
            if pantalla == 40: pantalla = 0


        elif pantalla == 2:
            nueva = 1
            flag = -1
            while nueva:
                clear_output(wait=True)
                time.sleep(0.05)
                print("Ingresar una palabra para ser agregada en la lista. Para terminar, no ingrese ningún caracter")
                if palabras_agregadas: print(f"Lista actual: {palabras_agregadas}\n")

                if flag == 1: print("Palabra invalida\n\n")
                elif flag == 2: print("La palabra debe estar entre 3 a 12 caracteres\n\n")
                elif flag == 0: print("Palabra agregada con exito!\n\n") 
                else: print()


                nueva = input("Ingrese la nueva palabra: ").strip()
                if not all(c.lower() in LETRAS_VALIDAS for c in nueva): flag = 1
                elif not (3<=len(nueva)<=12): flag = 2
                else:
                    palabras_agregadas.add(nueva.capitalize())
                    flag = 0
            pantalla = 0
        elif pantalla == 3: 
            clear_output(wait=True)
            time.sleep(0.05)
            print("README")
            input("\n\nPresione cualquier tecla para salir")
            pantalla = 0
        elif pantalla == 4: exit()

        elif 10<=pantalla<=30:
            clear_output(wait=True)
            time.sleep(0.05)
            print("1) Jugar en la terminal")
            print("2) Jugar con interfaz gráfica")
            print("3) Atras")
            while not (1<=pantalla<=3):
                pantalla = int(input("Opción: "))
            
            if pantalla == 1: return etapas,real, 1
            if pantalla == 2: return etapas,real, 2
            if pantalla == 3: pantalla = 1




etapas,real,num = consola()
if num == 1: 
    while True:
        terminal(etapas,real)
elif num == 2: pass
else: print("Error"); exit()
